In this the drivers data is in nested json structure and the nested structure is preserved

In [0]:
%run ../0-common/env-config

In [0]:
%run ../0-common/bronze_helpers

In [0]:
source_file = f"{landing_folfer_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

drivers__schema = StructType([
    StructField("driverId", StringType()),
    StructField("name", StructType([
        StructField("familyName", StringType()),
        StructField("givenName", StringType())
    ])),
     StructField("dateOfBirth", DateType()),
    StructField("nationality", StringType()),
    StructField("url", StringType())
])

In [0]:
drivers_df = (
    spark.read.format('json')
    .schema(drivers__schema)
    .option('mode', 'FAILFAST')
    .load(source_file)
)
display(drivers_df)

In [0]:
drivers_df_final = add_ingestion_metadata(drivers_df)

In [0]:
(
    drivers_df_final.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
SELECT * FROM formula1.bronze.drivers